# DH-PINN plate bending under a CENTERED RECTANGULAR PATCH ("partial uniform") load, (FFFF case).

In [2]:
# structure

In [1]:
"""
DH-PINN (Deterministic-Harmonic Fourier-feature input encoding + the
VANILLA-PINN backbone: Automatic Differentiation, NOT finite differences)
for Kirchhoff-Love plate bending under a PARTIALLY-DISTRIBUTED (centered
rectangular patch) load, all-FOUR-edges-CLAMPED / FIXED (FFFF).

>>> THIS FILE = the autograd-based FFFF + L-BFGS-polish script, with ONLY
>>> the network's INPUT BLOCK changed -- the same Deterministic-Harmonic
>>> Fourier-feature encoding added to DH-FDM-PINN, but here it sits on top
>>> of the ORIGINAL autograd backbone (torch.autograd.grad computes the
>>> exact symbolic derivatives of the network's output), not the
>>> finite-difference one. This is the ">>> VARIANT POINT: Fourier embed
>>> here for FF variants" placeholder from the earliest scripts, filled in
>>> on the Vanilla-PINN (autograd) backbone specifically. Everything else
>>> -- loading, hard-BC envelope, autograd PDE residual, analytical
>>> (beam-function Ritz) solution, Adam+L-BFGS training with min-delta
>>> early stopping, cost/error diagnostics, and plotting -- is UNCHANGED
>>> from the FFFF + L-BFGS-polish (autograd) script.

>>> DH FOURIER FEATURES (the new part, identical to DH-FDM-PINN):
>>> "Deterministic-Harmonic" means the embedding frequencies are a FIXED,
>>> non-trainable integer harmonic series 1, 2, ..., K (K = N_HARMONICS
>>> below) -- NOT randomly sampled (as in "Random Fourier Features" /
>>> Gaussian-RFF PINNs) and NOT learned. For each normalized coordinate
>>> xi = x/a, eta = y/b in [0,1], the encoding is
>>>     gamma(xi)  = [ sin(1*pi*xi), cos(1*pi*xi), ..., sin(K*pi*xi), cos(K*pi*xi) ]
>>>     gamma(eta) = [ sin(1*pi*eta), cos(1*pi*eta), ..., sin(K*pi*eta), cos(K*pi*eta) ]
>>> and the network input becomes [xi, eta, gamma(xi), gamma(eta)] (raw
>>> coordinates kept alongside the harmonics). This mirrors the natural
>>> Fourier/sine-series structure of the plate-bending problem itself (the
>>> SS-SS-SS-SS analytical solution IS a sin(m pi x/a) series; the FFFF one
>>> is a clamped-beam-function series built from the same harmonic family)
>>> -- feeding the network pre-computed harmonics up to order K gives it,
>>> for free, the same frequency building blocks the analytical series
>>> uses, counteracting the "spectral bias" that makes plain MLPs slow to
>>> learn high-frequency content. The frequencies are registered as a
>>> torch buffer (not an nn.Parameter), so they never receive a gradient
>>> and never change during training -- that's the "deterministic" half of
>>> the name; "harmonic" is the integer-multiple frequency choice.
>>>
>>> DH-PINN vs DH-FDM-PINN: identical input encoding, DIFFERENT residual
>>> mechanics. DH-PINN (this file) differentiates the network's OWN output
>>> exactly via nested torch.autograd.grad calls (no discretization error
>>> in the derivatives themselves, but each higher-order derivative adds
>>> another backward pass through the computation graph). DH-FDM-PINN
>>> estimates the same derivatives from neighboring grid-point VALUES via
>>> finite-difference stencils on a ghost-padded grid (one forward pass,
>>> but O(h^2) truncation error in the derivative estimates). Comparing
>>> the two under the SAME input encoding isolates the effect of the
>>> derivative-computation choice from the effect of the Fourier features.

Load definition (matches the user's figure: origin O at plate center, patch
of width u along x and v along y, centered on the plate, intensity q0
uniform inside the patch and zero outside):
    q(x,y) = q0   for |x - a/2| <= u/2  AND  |y - b/2| <= v/2
    q(x,y) = 0    otherwise
Patch size used here: u = 0.4*a, v = 0.4*b (PATCH_U_FRAC / PATCH_V_FRAC).

-----------------------------------------------------------------------------
ANALYTICAL SOLUTION -- DERIVATION NOTES (beam-function Rayleigh-Ritz/Galerkin,
clamped-clamped edges, i.e. FFFF)
-----------------------------------------------------------------------------
For SS-SS-SS-SS the sine functions sin(mpi x/a) are simultaneously (a) the
exact eigenfunctions of the 1-D operator d^4/dx^4 with SS end conditions and
(b) orthogonal, which is what makes the Navier series diagonal and closed
-form. For FFFF, the analogous 1-D building blocks are the mode shapes of a
CLAMPED-CLAMPED beam,
    X_m(x) = cosh(b_m x) - cos(b_m x) - sigma_m [sinh(b_m x) - sin(b_m x)],
    sigma_m = [cosh(lam_m) - cos(lam_m)] / [sinh(lam_m) - sin(lam_m)],
where b_m = lam_m/a and lam_m are the positive roots of
    cos(lam) cosh(lam) = 1        (lam_1=4.73004, lam_2=7.85320,
                                    lam_3=10.99561, lam_4=14.13717, ...).
X_m satisfies X_m(0)=X_m'(0)=X_m(a)=X_m'(a)=0 (clamped at both ends) and
X_m'''' = b_m^4 X_m, and the set {X_m} is orthogonal on [0,a] (self-adjoint
clamped-beam eigenproblem). The same construction gives Y_n(y) on [0,b].
Both are normalized here so that int_0^a X_m^2 dx = a (and likewise for Y_n).

Approximate the plate deflection as a double series in these (now BC-exact)
basis functions:
    w(x,y) = sum_{m=1}^{M} sum_{n=1}^{M} C_mn X_m(x) Y_n(y)
which satisfies w=0 AND dw/dn=0 on all four edges for ANY choice of C_mn --
i.e. the clamped essential BCs are built into the basis, exactly as the
sine series builds the simply-supported BC into its basis.

Galerkin-project the plate PDE D(w_xxxx + 2 w_xxyy + w_yyyy) = q onto the
same basis (equivalently: Rayleigh-Ritz minimization of the plate's bending
strain energy). Using X_m''''=b_m^4 X_m and integrating the cross term
w_xxyy by parts twice (boundary terms vanish because X_p=X_p'=0 at the
clamped ends, so no natural/moment BC ever enters -- consistent with the
fact that a clamped edge has no zero-moment condition to satisfy) gives the
fully-coupled (non-diagonal) linear system

    D [ C_pq b_p^4 a b + 2 sum_mn C_mn G_mp G_nq + C_pq a g_q^4 b ] = Q_pq

    G_mp = int_0^a X_m''(x) X_p(x) dx      (identical construction for Y)
    g_q  = b_q  (beam eigenvalue in the y-direction; here a=b so g=b_p)
    Q_pq = int_0^b int_0^a q(x,y) X_p(x) Y_q(y) dx dy
         = q0 * [int_{patch_x} X_p(x) dx] * [int_{patch_y} Y_q(y) dy]
           (patch load is separable in x and y, so Q_pq factors exactly
            like it did for the SS-SS-SS-SS Navier coefficients).

G_mp is NOT diagonal (unlike the SS-SS-SS-SS case) because clamped beam
modes of different order are orthogonal in {X_m, X_p} but not in
{X_m'', X_p}; this couples all (m,n) modes together into one M^2 x M^2
linear system, solved once with a dense linear solve (M=24 modes/direction
by default -> 576x576 system, solved in well under a second).

VERIFICATION (done before trusting this as ground truth, two independent
checks):
  1. Uniform-load limit (patch -> full plate) reproduces the standard
     tabulated clamped-square-plate benchmarks (Timoshenko &
     Woinowsky-Krieger, "all edges built in"):
       center deflection coefficient: computed 0.001266  vs tabulated 0.00126
       center Mx coefficient:         computed 0.02280   vs tabulated 0.0231
       edge mid-span Mx coefficient: computed -0.05124   vs tabulated -0.0513
  2. For the ACTUAL 0.4a x 0.4b patch load, cross-checked against an
     independent 13-point finite-difference biharmonic solver (clamped
     ghost-point BCs), grid-refined to 101x101:
       center deflection: Ritz 0.0006328  vs FD 0.0006329  (0.015% apart)
       center Mx:         Ritz 0.0147064  vs FD 0.0147147  (0.057% apart)
  Both checks agree to well under 0.1%, so this series is trusted as the
  FFFF ground truth here. M=24 is used as a converged operating point;
  raising N_MODES trades runtime for a further small tightening of the
  moment benchmarks (deflection converges fastest, then Mx/My, then Mxy).

Once C_mn is solved, moments follow the same way they did analytically for
SS-SS-SS-SS, just with the new basis:
    Mx(x,y)  = -D * sum C_mn [X_m''(x) Y_n(y) + nu X_m(x) Y_n''(y)]
    My(x,y)  = -D * sum C_mn [Y_n''(y) X_m(x) + nu Y_n''... ] (symmetric form)
    Mxy(x,y) = -D(1-nu) * sum C_mn X_m'(x) Y_n'(y)
-----------------------------------------------------------------------------
ALL FIXES CARRIED OVER FROM THE SS-SS-SS-SS / SINUSOIDAL / TRIANGULAR
SCRIPTS THAT ARE STILL APPLICABLE:
  2. PDE residual no longer divided by an arbitrary 100.
  4. SiLU activation.
  6. Mxy analytical formula's 1/(a*b)-type factor confirmed present (now via
     the beam-function series instead of the sine series).
  7-8. Field reshape/indexing uses (ny, nx) consistently (row=y, col=x).
  9. epochs_to_convergence renamed to epochs_to_early_stop (old key kept as
     deprecated alias).
 10. loss_balance.json diagnostic kept (trivially 100% pde_loss, since there
     is no soft moment loss term for FFFF).
 11. HARD BC: a clamped edge requires BOTH w=0 and dw/dn=0, i.e. two
     essential (Dirichlet-type) conditions, and NO natural (moment)
     condition. Squaring each 1-D envelope factor,
         D(x,y) = [(x/a)(1-x/a)]^2 * [(y/b)(1-y/b)]^2,
     keeps D=0 on every edge, and now dD/dx (and dD/dy) also vanish on
     every edge, because d/dx[xi^2(1-xi)^2] = 2xi(1-xi)(1-2xi) which is 0 at
     xi=0 and xi=1 too. So w(x,y) = scale * D(x,y) * N(x,y) hard-enforces
     w=0 AND dw/dn=0 on all four edges exactly by construction -- no soft
     moment/slope loss term is needed at all. bc_loss and slope_loss below
     are diagnostics only; their gradients are identically zero.

NOTE ON THE PDE FORCING TERM: the patch load is a hard rectangular
indicator function, evaluated forward-only (never differentiated) in the
PDE residual. The 51x51 collocation grid resolves the u=v=0.4 patch with
~20 points per side.
-----------------------------------------------------------------------------

Outputs (all written to ./results/<MODEL_NAME>_<LOADING_CASE>_<BC_TYPE>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python dh_pinn_partial_patch_hardBC_FFFF.py
"""

import os
import json
import time
import random
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import brentq
from scipy.integrate import quad

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "DH_PINN"                # >>> CHANGED (was "Vanilla_PINN_HardBC") -- Deterministic-Harmonic Fourier input encoding added
LOADING_CASE = "partial_patch"       # centered rectangular patch load
BC_TYPE = "FFFF"                     # all four edges fixed/clamped
N_MODES = 24                         # beam-function modes per direction (see derivation notes)
N_HARMONICS = 8                      # >>> NEW: deterministic harmonic Fourier-feature order K (frequencies 1..K, fixed, non-trainable)

# Patch size as a fraction of the plate dimensions (confirmed with user: 0.4 x 0.4)
PATCH_U_FRAC = 0.4   # patch width along x, as a fraction of a
PATCH_V_FRAC = 0.4   # patch height along y, as a fraction of b

# NOTE: OUT_DIR includes LOADING_CASE and BC_TYPE so results don't overwrite
# the SS-SS-SS-SS results from the other script.
OUT_DIR = os.path.join("results", f"{MODEL_NAME}_{LOADING_CASE}_{BC_TYPE}")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

# --- FIX B: min-delta early stopping ------------------------------------
# A new "best" now requires at least this RELATIVE improvement over the
# previous best, so noisy micro-improvements no longer reset the patience
# counter and mask a genuine plateau.
MIN_DELTA_REL = 1e-4

# --- FIX C: L-BFGS polish phase ------------------------------------------
LBFGS_MAX_ITER = 500          # internal L-BFGS iterations per outer step
LBFGS_OUTER_STEPS = 4         # how many times to call optimizer.step(closure)
LBFGS_HISTORY_SIZE = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

PATCH_U = PATCH_U_FRAC * a_nd   # patch width along x (non-dim)
PATCH_V = PATCH_V_FRAC * b_nd   # patch height along y (non-dim)

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept only for the diagnostic bc_loss now ------------
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices, kept only for the diagnostic SLOPE residual below.
# (For FFFF there is no soft moment BC -- both w=0 and dw/dn=0 are hard-
# enforced by the network's envelope, see PlatePINN.forward.)
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # w_x = 0 here (slope diag.)
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # w_y = 0 here (slope diag.)

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# ANALYTICAL SOLUTION -- beam-function Rayleigh-Ritz/Galerkin, FFFF
# (clamped-clamped in both directions). See derivation notes in the module
# docstring; verified there against Timoshenko & Woinowsky-Krieger's
# tabulated clamped-square-plate benchmarks AND an independent
# finite-difference solve of the actual patch-load case.
# ============================================================

def _clamped_beam_eigenvalues(n_modes):
    """Positive roots lam_m of cos(lam)cosh(lam) = 1 (clamped-clamped beam)."""
    def f(lam):
        return np.cos(lam) * np.cosh(lam) - 1.0
    lams = []
    k = 1
    while len(lams) < n_modes:
        # brackets centered on the large-lam asymptote lam_m -> (2m+1)pi/2
        lo = (2 * k + 1) * np.pi / 2 - 1.5 if k > 1 else 4.0
        hi = (2 * k + 1) * np.pi / 2 + 1.5 if k > 1 else 5.6
        try:
            root = brentq(f, lo, hi)
            if len(lams) == 0 or root - lams[-1] > 1e-6:
                lams.append(root)
        except ValueError:
            pass
        k += 1
    return np.array(lams[:n_modes])


_lam = _clamped_beam_eigenvalues(N_MODES)
_beta = _lam / a_nd          # x-direction beam wavenumbers (a_nd = b_nd = 1 here)
_beta_y = _lam / b_nd        # y-direction beam wavenumbers
_sigma = (np.cosh(_lam) - np.cos(_lam)) / (np.sinh(_lam) - np.sin(_lam))


def _Xraw(m, x):
    bm = _beta[m]
    return np.cosh(bm * x) - np.cos(bm * x) - _sigma[m] * (np.sinh(bm * x) - np.sin(bm * x))


def _Xraw_p(m, x):
    bm = _beta[m]
    return bm * (np.sinh(bm * x) + np.sin(bm * x) - _sigma[m] * (np.cosh(bm * x) - np.cos(bm * x)))


def _Xraw_pp(m, x):
    bm = _beta[m]
    return bm ** 2 * (np.cosh(bm * x) + np.cos(bm * x) - _sigma[m] * (np.sinh(bm * x) + np.sin(bm * x)))


def _Yraw(n, y):
    bn = _beta_y[n]
    return np.cosh(bn * y) - np.cos(bn * y) - _sigma[n] * (np.sinh(bn * y) - np.sin(bn * y))


def _Yraw_p(n, y):
    bn = _beta_y[n]
    return bn * (np.sinh(bn * y) + np.sin(bn * y) - _sigma[n] * (np.cosh(bn * y) - np.cos(bn * y)))


def _Yraw_pp(n, y):
    bn = _beta_y[n]
    return bn ** 2 * (np.cosh(bn * y) + np.cos(bn * y) - _sigma[n] * (np.sinh(bn * y) + np.sin(bn * y)))


# Normalize so that int_0^a X_m^2 dx = a (and likewise for Y_n over b).
_norm_x = np.array([
    np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
    for m in range(N_MODES)
])
_norm_y = np.array([
    np.sqrt(quad(lambda y, n=n: _Yraw(n, y) ** 2, 0, b_nd, limit=200)[0] / b_nd)
    for n in range(N_MODES)
])


def _X(m, x):
    return _Xraw(m, x) / _norm_x[m]


def _Xp(m, x):
    return _Xraw_p(m, x) / _norm_x[m]


def _Xpp(m, x):
    return _Xraw_pp(m, x) / _norm_x[m]


def _Y(n, y):
    return _Yraw(n, y) / _norm_y[n]


def _Yp(n, y):
    return _Yraw_p(n, y) / _norm_y[n]


def _Ypp(n, y):
    return _Yraw_pp(n, y) / _norm_y[n]


# G_mp = int_0^a X_m''(x) X_p(x) dx  (NOT diagonal -- couples all mode pairs)
_G = np.zeros((N_MODES, N_MODES))
for _m in range(N_MODES):
    for _p in range(N_MODES):
        _G[_m, _p] = quad(lambda x, m=_m, p=_p: _Xpp(m, x) * _X(p, x), 0, a_nd, limit=200)[0]

# H_nq analogous in y (identical construction here since a_nd == b_nd)
_H = np.zeros((N_MODES, N_MODES))
for _n in range(N_MODES):
    for _q in range(N_MODES):
        _H[_n, _q] = quad(lambda y, n=_n, q=_q: _Ypp(n, y) * _Y(q, y), 0, b_nd, limit=200)[0]

_beta4 = _beta ** 4
_gamma4 = _beta_y ** 4


def _idx(p, q):
    return p * N_MODES + q


def _assemble_and_solve_patch_load(u_frac, v_frac, q0):
    """Solve the FFFF Rayleigh-Ritz linear system for a centered patch load
    of size (u_frac*a) x (v_frac*b), intensity q0. Returns C[m, n]."""
    half_u, half_v = u_frac * a_nd / 2, v_frac * b_nd / 2
    x_lo, x_hi = a_nd / 2 - half_u, a_nd / 2 + half_u
    y_lo, y_hi = b_nd / 2 - half_v, b_nd / 2 + half_v

    Ip = np.array([quad(lambda x, p=p: _X(p, x), x_lo, x_hi, limit=200)[0] for p in range(N_MODES)])
    Iq = np.array([quad(lambda y, q=q: _Y(q, y), y_lo, y_hi, limit=200)[0] for q in range(N_MODES)])

    Npq = N_MODES * N_MODES
    K = np.zeros((Npq, Npq))
    Qvec = np.zeros(Npq)
    for p in range(N_MODES):
        for q in range(N_MODES):
            row = _idx(p, q)
            K[row, _idx(p, q)] += D_nd * (_beta4[p] * a_nd * b_nd + a_nd * _gamma4[q] * b_nd)
            # cross (twist/coupling) term -- couples every (m,n) into every (p,q)
            K[row, :] += 2 * D_nd * np.einsum("m,n->mn", _G[:, p], _H[:, q]).ravel()
            Qvec[row] = q0 * Ip[p] * Iq[q]

    C = np.linalg.solve(K, Qvec).reshape(N_MODES, N_MODES)
    return C


# Solved once, module-level (mirrors the SS-SS-SS-SS script's precomputed
# _q_mn / _W_mn tables). See module docstring for the benchmark checks.
_C_mn = _assemble_and_solve_patch_load(PATCH_U_FRAC, PATCH_V_FRAC, q0_nd)


def _series_field(x, y, kind="w"):
    """Evaluate the beam-function Ritz series (w, Mx, My, or Mxy) on a field
    grid x, y (numpy arrays, any shape)."""
    out = np.zeros_like(x, dtype=float)
    for m in range(N_MODES):
        Xm = _X(m, x)
        Xmp = _Xp(m, x)
        Xmpp = _Xpp(m, x)
        for n in range(N_MODES):
            c = _C_mn[m, n]
            if c == 0.0:
                continue
            if kind == "w":
                out += c * Xm * _Y(n, y)
            elif kind == "Mx":
                out += -D_nd * c * (Xmpp * _Y(n, y) + nu_nd * Xm * _Ypp(n, y))
            elif kind == "My":
                out += -D_nd * c * (_Ypp(n, y) * Xm + nu_nd * Xmpp * _Y(n, y))
            elif kind == "Mxy":
                out += -D_nd * (1 - nu_nd) * c * Xmp * _Yp(n, y)
            else:
                raise ValueError(kind)
    return out


def analytical_w_nd(x, y):
    return _series_field(x, y, "w")


def analytical_Mx_nd(x, y):
    return _series_field(x, y, "Mx")


def analytical_My_nd(x, y):
    return _series_field(x, y, "My")


def analytical_Mxy_nd(x, y):
    return _series_field(x, y, "Mxy")


def partial_patch_load_nd(x, y):
    """Uniform load q0 over a centered rectangular patch of size
    PATCH_U x PATCH_V, zero elsewhere. x, y are torch tensors (unchanged
    from the SS-SS-SS-SS script -- the load itself doesn't depend on BC)."""
    half_u = PATCH_U / 2.0
    half_v = PATCH_V / 2.0
    in_x = (x >= (a_nd / 2 - half_u)) & (x <= (a_nd / 2 + half_u))
    in_y = (y >= (b_nd / 2 - half_v)) & (y <= (b_nd / 2 + half_v))
    mask = (in_x & in_y).to(DTYPE)
    return q0_nd * mask


# ============================================================
# MODEL -- >>> CHANGED (input block only): raw (xi, eta) is now expanded
# into deterministic-harmonic Fourier features before the hidden layers.
# This is the ">>> VARIANT POINT: Fourier embed here for FF variants" from
# the original comment, now filled in. The hidden-layer stack, hard-BC
# envelope, and the AUTOGRAD residual mechanics in compute_loss() below
# (which only require the model to be a differentiable function of x,y --
# sin/cos are differentiable, so nothing else needs to change) are all
# UNCHANGED from the Vanilla-PINN FFFF script.
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3, n_harmonics=N_HARMONICS):
        super().__init__()
        self.n_harmonics = n_harmonics

        # >>> NEW: deterministic harmonic frequencies 1, 2, ..., K. Registered
        # as a buffer (not nn.Parameter), so they are fixed for the entire
        # training run -- never receive a gradient, never get updated by
        # the optimizer (Adam or L-BFGS). That's the "deterministic" half
        # of the name; "harmonic" is the integer-multiple frequency choice
        # (as opposed to randomly-sampled or learned frequencies).
        freqs = torch.arange(1, n_harmonics + 1, dtype=DTYPE)
        self.register_buffer("freqs", freqs)

        # Input dim: raw (xi, eta) = 2, plus [sin,cos] of each harmonic for
        # BOTH xi and eta = 4*K. Raw coordinates are kept alongside the
        # harmonics (sin/cos features alone are poor at representing the
        # low-frequency/linear trend near a boundary).
        in_dim = 2 + 4 * n_harmonics
        layers = []
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def _fourier_encode(self, xi, eta):
        """xi, eta: (N,1) tensors in [0,1]. Returns (N, 2+4K):
        [xi, eta, sin(k*pi*xi)_k, cos(k*pi*xi)_k, sin(k*pi*eta)_k, cos(k*pi*eta)_k].
        Every op here (mul, sin, cos, cat) is a standard differentiable
        torch op, so autograd.grad w.r.t. the ORIGINAL x, y in compute_loss()
        still flows through this layer correctly via the chain rule --
        nothing about the autograd residual mechanics needs to change."""
        x_phase = xi * self.freqs.view(1, -1) * math.pi   # (N, K)
        y_phase = eta * self.freqs.view(1, -1) * math.pi  # (N, K)
        return torch.cat([
            xi, eta,
            torch.sin(x_phase), torch.cos(x_phase),
            torch.sin(y_phase), torch.cos(y_phase),
        ], dim=1)

    def forward(self, x, y):
        xi = x / a_nd
        eta = y / b_nd

        h = self._fourier_encode(xi, eta)   # >>> CHANGED: was torch.cat([xi, eta], dim=1)
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained essential BCs for FFFF: a clamped edge needs
        # w=0 AND dw/dn=0. Squaring each 1-D factor keeps w=0 on every edge
        # and additionally makes the normal derivative vanish there too,
        # since d/dxi[xi^2(1-xi)^2] = 2 xi(1-xi)(1-2xi) = 0 at xi=0, xi=1.
        envelope = (xi * (1 - xi)) ** 2 * (eta * (1 - eta)) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE (Adam phase)
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via autograd)
# ============================================================
def compute_loss():
    x = X_grid[:, 0:1].clone().requires_grad_(True)
    y = X_grid[:, 1:2].clone().requires_grad_(True)

    w = model(x, y)
    ones = torch.ones_like(w)

    w_x = torch.autograd.grad(w, x, grad_outputs=ones, create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=ones, create_graph=True)[0]

    w_xx = torch.autograd.grad(w_x, x, grad_outputs=ones, create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=ones, create_graph=True)[0]

    w_xxx = torch.autograd.grad(w_xx, x, grad_outputs=ones, create_graph=True)[0]
    w_yyy = torch.autograd.grad(w_yy, y, grad_outputs=ones, create_graph=True)[0]
    w_xxy = torch.autograd.grad(w_xx, y, grad_outputs=ones, create_graph=True)[0]

    w_xxxx = torch.autograd.grad(w_xxx, x, grad_outputs=ones, create_graph=True)[0]
    w_yyyy = torch.autograd.grad(w_yyy, y, grad_outputs=ones, create_graph=True)[0]
    w_xxyy = torch.autograd.grad(w_xxy, y, grad_outputs=ones, create_graph=True)[0]

    q = partial_patch_load_nd(x, y)

    pde = (w_xxxx + 2 * w_xxyy + w_yyyy - q)
    pde_loss = torch.mean(pde[int_idx] ** 2)

    # DIAGNOSTIC ONLY -- both should be ~0 exactly, by construction of the
    # hard-BC envelope. Neither is included in `total`; their gradients are
    # identically zero anyway.
    bc_loss = torch.mean(w[bc_idx] ** 2)                       # deflection residual
    slope_loss = (torch.mean(w_x[edge_x_idx] ** 2)
                  + torch.mean(w_y[edge_y_idx] ** 2))           # normal-slope residual

    # No soft moment loss for FFFF -- a clamped edge has no zero-moment
    # (natural) condition, so the training loss is the PDE residual alone.
    total = pde_loss
    return total, pde_loss, bc_loss, slope_loss, w


# ============================================================
# ADAM TRAINING LOOP with computational-cost logging + min-delta early
# stopping (FIX B)
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_history = [], [], [], [], []
w_center_history, epoch_time_history, phase_history = [], [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS
adam_epochs_run = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope, w_pred = compute_loss()
    total.backward()
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_history.append(slope.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)
    phase_history.append("adam")

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}, "
              f"hardBC_w_residual {bc.item():.3e}, hardBC_slope_residual {slope.item():.3e}) "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    # FIX B: relative min-delta improvement, not any-improvement
    if loss_val < best_loss * (1.0 - MIN_DELTA_REL):
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping (Adam phase) triggered at epoch {epoch}")
            converged_epoch = epoch
            adam_epochs_run = epoch + 1
            break

adam_time = time.perf_counter() - train_start

# Make sure a checkpoint exists even if the very first epoch was already
# "best" and nothing since improved on it enough to re-save.
if not os.path.exists(weights_path):
    torch.save(model.state_dict(), weights_path)

# ============================================================
# FIX C: L-BFGS POLISH PHASE
# Reload the best Adam checkpoint, then refine with second-order L-BFGS
# steps (strong-Wolfe line search) on the SAME loss. Each closure call is
# logged into the same history arrays (phase="lbfgs") so the convergence
# CSV and loss-curve plot show both phases continuously.
# ============================================================
model.load_state_dict(torch.load(weights_path, map_location=DEVICE))

lbfgs_optimizer = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=LBFGS_MAX_ITER,
    history_size=LBFGS_HISTORY_SIZE,
    line_search_fn="strong_wolfe",
)

lbfgs_start = time.perf_counter()
lbfgs_best_loss = best_loss
lbfgs_iters_run = 0


def _lbfgs_closure():
    global lbfgs_best_loss, lbfgs_iters_run
    lbfgs_optimizer.zero_grad()
    total, pde, bc, slope, w_pred = compute_loss()
    total.backward()

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()
    epoch_history.append(adam_epochs_run + lbfgs_iters_run)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_history.append(slope.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(float("nan"))  # per-iter timing not meaningful inside LBFGS closure
    phase_history.append("lbfgs")

    if loss_val < lbfgs_best_loss:
        lbfgs_best_loss = loss_val
        torch.save(model.state_dict(), weights_path)

    lbfgs_iters_run += 1
    return total


for outer_step in range(LBFGS_OUTER_STEPS):
    lbfgs_optimizer.step(_lbfgs_closure)
    print(f"L-BFGS outer step {outer_step + 1}/{LBFGS_OUTER_STEPS} done "
          f"(best loss so far: {lbfgs_best_loss:.6e})")

lbfgs_time = time.perf_counter() - lbfgs_start
total_train_time = adam_time + lbfgs_time
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV (Adam + L-BFGS phases)
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "phase": phase_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_w_residual_diagnostic": bc_history,
    "hard_bc_slope_residual_diagnostic": slope_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC (trivially 100% pde_loss for FFFF -- no soft
# moment term exists to weight, unlike SS-SS-SS-SS)
# ============================================================
final_pde, final_bc, final_slope = pde_history[-1], bc_history[-1], slope_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "final_pde_loss": final_pde,
    "pde_fraction_of_total": 1.0,
    "final_hard_bc_w_residual_diagnostic": final_bc,
    "final_hard_bc_slope_residual_diagnostic": final_slope,
}
if final_bc > 1e-20 or final_slope > 1e-20:
    print(f"WARNING: hard BC residual diagnostics are not ~0 "
          f"(w: {final_bc:.3e}, slope: {final_slope:.3e}) -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
adam_epoch_times = [t for t, p in zip(epoch_time_history, phase_history) if p == "adam"]
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "device": str(DEVICE),
    "adam_epochs_run": adam_epochs_run,
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "lbfgs_outer_steps": LBFGS_OUTER_STEPS,
    "lbfgs_max_iter_per_step": LBFGS_MAX_ITER,
    "lbfgs_closure_calls": lbfgs_iters_run,
    "mean_time_per_adam_epoch_s": float(np.mean(adam_epoch_times)) if adam_epoch_times else float("nan"),
    "median_time_per_adam_epoch_s": float(np.median(adam_epoch_times)) if adam_epoch_times else float("nan"),
    "adam_phase_time_s": adam_time,
    "lbfgs_phase_time_s": lbfgs_time,
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My) via autograd
# ============================================================
def compute_full_fields():
    x = X_grid[:, 0:1].clone().requires_grad_(True)
    y = X_grid[:, 1:2].clone().requires_grad_(True)
    w = model(x, y)
    ones = torch.ones_like(w)

    w_x = torch.autograd.grad(w, x, grad_outputs=ones, create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=ones, create_graph=True)[0]
    w_xx = torch.autograd.grad(w_x, x, grad_outputs=ones, create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=ones, create_graph=True)[0]
    w_xy = torch.autograd.grad(w_x, y, grad_outputs=ones, create_graph=True)[0]

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mxy = -(1 - nu_nd) * w_xy

    w_grid = w.detach().cpu().numpy().reshape(ny, nx)
    Mx_grid = Mx.detach().cpu().numpy().reshape(ny, nx)
    My_grid = My.detach().cpu().numpy().reshape(ny, nx)
    Mxy_grid = Mxy.detach().cpu().numpy().reshape(ny, nx)
    return w_grid, Mx_grid, My_grid, Mxy_grid


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
lbfgs_start_epoch = adam_epochs_run
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Analytical")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.yscale("log")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.legend(); plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_bc_w_residual_diagnostic"],
         label="Hard-BC w residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["hard_bc_slope_residual_diagnostic"],
         label="Hard-BC slope residual (diagnostic, ~0)", alpha=0.5, linestyle=":")
plt.axvline(x=lbfgs_start_epoch, color="gray", linestyle=":", linewidth=1, label="L-BFGS starts")
plt.yscale("log")
plt.xlabel("Epoch (Adam) / Closure call (L-BFGS)"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "Analytical Plate Deflection", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Analytical Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Analytical Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Analytical Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Analytical displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Analytical vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_TYPE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Analytical {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading (u={PATCH_U_FRAC}a, v={PATCH_V_FRAC}b), "
             f"{BC_TYPE} (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Adam phase: {adam_time:.2f}s ({adam_epochs_run} epochs, early-stop epoch "
      f"{converged_epoch}) | L-BFGS phase: {lbfgs_time:.2f}s ({lbfgs_iters_run} closure calls) | "
      f"Total: {total_train_time:.2f}s | Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Adam phase: {adam_time:.2f}s ({adam_epochs_run} epochs) | "
      f"L-BFGS phase: {lbfgs_time:.2f}s ({lbfgs_iters_run} closure calls) | "
      f"Total: {total_train_time:.2f}s")

C:\Users\vivek\AppData\Local\Temp\ipykernel_5608\316949172.py:356: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
C:\Users\vivek\AppData\Local\Temp\ipykernel_5608\316949172.py:356: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  np.sqrt(quad(lambda x, m=m: _Xraw(m, x) ** 2, 0, a_nd, limit=200)[0] / a_nd)
C:\Users\vivek\AppData\Local\Temp\ipykernel_5608\316949172.py:360: IntegrationWarnin

Epoch 0 Loss 3.276258e+01 (pde 3.276e+01, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 2.006624e-04 (635.97 ms/epoch)
Epoch 100 Loss 7.224064e-02 (pde 7.224e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 4.587863e-04 (204.76 ms/epoch)
Epoch 200 Loss 5.394181e-02 (pde 5.394e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 5.008948e-04 (205.61 ms/epoch)
Epoch 300 Loss 4.550829e-02 (pde 4.551e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 5.108025e-04 (246.92 ms/epoch)
Epoch 400 Loss 4.072177e-02 (pde 4.072e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 5.172120e-04 (247.36 ms/epoch)
Epoch 500 Loss 3.736306e-02 (pde 3.736e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 5.211646e-04 (243.84 ms/epoch)
Epoch 600 Loss 3.457354e-02 (pde 3.457e-02, hardBC_w_residual 0.000e+00, hardBC_slope_residual 0.000e+00) Center 5.239145e-04 (266.48 ms/e

C:\Users\vivek\AppData\Local\Temp\ipykernel_5608\316949172.py:689: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=

L-BFGS outer step 1/4 done (best loss so far: 2.128448e-04)
L-BFGS outer step 2/4 done (best loss so far: 1.484707e-04)
L-BFGS outer step 3/4 done (best loss so far: 1.125717e-04)
L-BFGS outer step 4/4 done (best loss so far: 9.113030e-05)


C:\Users\vivek\AppData\Local\Temp\ipykernel_5608\316949172.py:738: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'DH_PINN', 'loading_case': 'partial_patch', 'bc_type': 'FFFF', 'final_pde_loss': 9.113029570605047e-05, 'pde_fraction_of_total': 1.0, 'final_hard_bc_w_residual_diagnostic': 0.0, 'final_hard_bc_slope_residual_diagnostic': 0.0}
Saved cost_metrics.json: {'model': 'DH_PINN', 'loading_case': 'partial_patch', 'bc_type': 'FFFF', 'device': 'cuda', 'adam_epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs_to_convergence': 15000, 'lbfgs_outer_steps': 4, 'lbfgs_max_iter_per_step': 500, 'lbfgs_closure_calls': 2136, 'mean_time_per_adam_epoch_s': 0.28492835110025483, 'median_time_per_adam_epoch_s': 0.28497194999363273, 'adam_phase_time_s': 4292.809986099834, 'lbfgs_phase_time_s': 655.5480927000754, 'total_training_time_s': 4948.358078799909, 'peak_gpu_memory_mb': 1198.065664, 'n_trainable_params': 10626}
Saved error_metrics.json: {'model': 'DH_PINN', 'loading_case': 'partial_patch', 'bc_type': 'FFFF', 'center_deflection_pred_m'